# Coordinator - Sub-agent Notebook

In this notebook we will follow the build exercise from https://claudecertificationguide.com/learn/1-agentic-architecture/1-2-orchestration-patterns

* Coordinator agent: sits at the centre. Receives the initial task, decomposes it, decides which subagents to invoke, passes context to them, aggregates their results, handles errors, and routes information between them.
* Subagents: the spokes. Each one handles a specialised task (web search, document analysis, synthesis, report generation). They receive instructions from the coordinator and return results to it.

```mermaid
flowchart TD
    User([User / Initial Task]) --> Coord{{Coordinator Agent}}
    Coord -->|"1. Decompose task"| Coord
    Coord -->|"2. Delegate + pass context"| SA1[Subagent: Web Search]
    Coord -->|"2. Delegate + pass context"| SA2[Subagent: Document Analysis]
    Coord -->|"2. Delegate + pass context"| SA3[Subagent: Synthesis]
    Coord -->|"2. Delegate + pass context"| SA4[Subagent: Report Generation]
    SA1 -->|"3. Return result"| Coord
    SA2 -->|"3. Return result"| Coord
    SA3 -->|"3. Return result"| Coord
    SA4 -->|"3. Return result"| Coord
    Coord -->|"4. Aggregate results, handle errors, route between subagents"| Coord
    Coord --> Output([Final Response])
```

## Coordinator Responsibilities

The coordinator has four key responsibilities that the exam tests:

| # | Responsibility | What it means |
|---|---|---|
| 1 | **Dynamic subagent selection** | Analyses query requirements and dynamically selects which subagents to invoke. It does **not** always route through the full pipeline — a simple factual question might only need the web search subagent, not the full research → analysis → synthesis chain. Routing every query through every subagent wastes time and resources. |
| 2 | **Research scope partitioning** | When delegating to multiple subagents, partitions the research scope to minimise duplication by assigning distinct subtopics or source types to each agent (e.g. one agent searches academic papers, another searches news articles — they don't both search the same sources). |
| 3 | **Iterative refinement loops** | Evaluates synthesis output for gaps. If the synthesis is incomplete, re-delegates to search and analysis subagents with targeted queries and re-invokes synthesis until coverage is sufficient. This is **not** a single-shot process — it's an iterative cycle. |
| 4 | **Centralised communication routing** | All subagent communication routes through the coordinator, enabling observability, consistent error handling, and controlled information flow. |

### Observability 
* you can log and monitor every message in one place.
### Consistent error handling 
* the coordinator applies uniform error recovery policies.
### Controlled information flow 
* the coordinator decides what context each subagent receives.

Subagents do NOT automatically inherit the coordinator's conversation history.

# What you'll learn

* How hub-and-spoke architecture centralises all communication through a coordinator
* Why subagent isolation means every piece of context must be explicitly passed
* How to implement broad task decomposition that avoids the narrow decomposition failure
* How iterative refinement loops detect and fill coverage gaps
* Why tracing failures to the coordinator decomposition is the correct diagnostic approach

# Step 1: Create a coordinator agent that accepts a broad research topic

In [ ]:
import os
from anthropic import AsyncAnthropic
from dotenv import load_dotenv

load_dotenv()

In [ ]:
client = AsyncAnthropic(api_key=os.environ["ANTHROPIC_KEY"])

In [7]:
import asyncio
import json
def print_json(label: str, data) -> None:
    """Pretty-print a plain value or an Anthropic SDK object (pydantic model) as JSON."""

    def default(o):
        if hasattr(o, "model_dump"):
            return o.model_dump(mode="json")
        return str(o)

    print(f"\n--- {label} ---")
    print(json.dumps(data, indent=2, default=default))

In [8]:
# Step 1: the user's request to the coordinator.
messages = [
    {"role": "user", "content": "Please give me some predictions about the oil market right now"},
]
print_json("Step 1: user request to the coordinator", messages)


--- Step 1: user request to the coordinator ---
[
  {
    "role": "user",
    "content": "Please give me some predictions about the oil market right now"
  }
]


In [10]:
 systemPrompt = """You are a research coordinator. Decompose topics into comprehensive subtopics, 
 delegate to specialist subagents, aggregate results, and identify coverage gaps."""

In [11]:
response = await client.messages.create(
    model="claude-sonnet-5",
    max_tokens=1024,
    messages=messages,
    system_prompt = systemPrompt
)
print_json("Step 2: response.content from first API call", response.content)

TypeError: AsyncMessages.create() got an unexpected keyword argument 'system_prompt'